# Trade Bot - LSTM Training

Entraînement LSTM multi-horizon pour prédiction direction SOL/USDC.

**Dataset:** 2.5M lignes, 70 features (5 cryptos), 2021→2026

**Modèles:** 3 LSTM spécialisés (5min, 15min, 1h) + méta-modèle

**Input:** Fenêtre glissante 60min × 70 features

**Output:** Classification binaire (SOL monte=1 / baisse=0)

## 1. Setup & Auth GCS

In [ ]:
from google.colab import auth
auth.authenticate_user()

!gcloud config set project trade-ml-bot

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import os, gzip, io

print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')

## 2. Charger les données depuis GCS

In [ ]:
BUCKET = 'gs://trade-ml-bot-data'

# Download from GCS
!gsutil cp {BUCKET}/dataset_wide_5.gz /tmp/dataset_wide_5.gz
!gsutil cp {BUCKET}/dataset_wide_5_labels.gz /tmp/dataset_wide_5_labels.gz
!gsutil cp {BUCKET}/dataset_wide_5_norm.csv.gz /tmp/dataset_wide_5_norm.csv.gz

print('Download OK')

In [ ]:
# Load datasets
print('Loading dataset...')
df = pd.read_csv('/tmp/dataset_wide_5.gz', compression='gzip')
print(f'Dataset: {df.shape}')

print('Loading labels...')
labels = pd.read_csv('/tmp/dataset_wide_5_labels.gz', compression='gzip')
print(f'Labels: {labels.shape}')

print('Loading norm stats...')
norm = pd.read_csv('/tmp/dataset_wide_5_norm.csv.gz', compression='gzip')
print(f'Norm: {norm.shape}')

df.head()

## 3. Préparer les features

In [ ]:
# Colonnes features (tout sauf 'time')
feature_cols = [c for c in df.columns if c != 'time']
print(f'{len(feature_cols)} features: {feature_cols[:10]}...')

# Normaliser avec mean/std pré-calculés
norm_dict = norm.set_index('column')
for col in feature_cols:
    if col in norm_dict.index:
        mean = norm_dict.loc[col, 'mean']
        std = norm_dict.loc[col, 'std']
        if std > 0:
            df[col] = (df[col] - mean) / std

# Remplacer NaN par 0
df[feature_cols] = df[feature_cols].fillna(0)

print('Normalisation OK')
df[feature_cols].describe()

In [ ]:
# Merger dataset + labels sur time
df['time'] = pd.to_datetime(df['time'])
labels['time'] = pd.to_datetime(labels['time'])

merged = df.merge(labels[['time', 'label_5m', 'label_15m', 'label_60m']], on='time', how='inner')
merged = merged.sort_values('time').reset_index(drop=True)

# Supprimer lignes sans label
merged = merged.dropna(subset=['label_5m', 'label_15m', 'label_60m'])

print(f'Merged: {merged.shape}')
print(f'Label distribution 15m: {merged["label_15m"].value_counts().to_dict()}')

## 4. Construire les fenêtres glissantes

In [ ]:
WINDOW_SIZE = 60  # 60 minutes d'input

# Convertir en numpy pour la vitesse
features_np = merged[feature_cols].values.astype(np.float32)
labels_5m = merged['label_5m'].values.astype(np.float32)
labels_15m = merged['label_15m'].values.astype(np.float32)
labels_60m = merged['label_60m'].values.astype(np.float32)

n_samples = len(features_np) - WINDOW_SIZE
n_features = features_np.shape[1]

print(f'Samples: {n_samples:,}')
print(f'Features per step: {n_features}')
print(f'Window shape: ({WINDOW_SIZE}, {n_features})')
print(f'Total tensor: ({n_samples}, {WINDOW_SIZE}, {n_features}) = {n_samples * WINDOW_SIZE * n_features * 4 / 1e9:.1f} GB')

In [ ]:
# Créer les fenêtres avec stride_tricks (mémoire efficace)
from numpy.lib.stride_tricks import sliding_window_view

# sliding_window_view crée une vue, pas une copie
X_windows = sliding_window_view(features_np, WINDOW_SIZE, axis=0)
# Shape: (n_samples, n_features, WINDOW_SIZE) → transpose to (n_samples, WINDOW_SIZE, n_features)
X_windows = np.moveaxis(X_windows, -1, 1)

# Labels: on prend le label à la FIN de chaque fenêtre
y_5m = labels_5m[WINDOW_SIZE:]
y_15m = labels_15m[WINDOW_SIZE:]
y_60m = labels_60m[WINDOW_SIZE:]

# Vérifier qu'on a les mêmes tailles
assert len(X_windows) == len(y_15m), f'{len(X_windows)} != {len(y_15m)}'

print(f'X: {X_windows.shape}')
print(f'y_5m: {y_5m.shape}, y_15m: {y_15m.shape}, y_60m: {y_60m.shape}')

## 5. Split train/val/test (chronologique)

In [ ]:
# Split chronologique: 70% train, 15% val, 15% test
n = len(X_windows)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_train = X_windows[:train_end]
X_val = X_windows[train_end:val_end]
X_test = X_windows[val_end:]

# On commence avec l'horizon 15min
y_train = y_15m[:train_end]
y_val = y_15m[train_end:val_end]
y_test = y_15m[val_end:]

print(f'Train: {X_train.shape} ({y_train.mean():.3f} positifs)')
print(f'Val:   {X_val.shape} ({y_val.mean():.3f} positifs)')
print(f'Test:  {X_test.shape} ({y_test.mean():.3f} positifs)')

# Dates correspondantes
times = merged['time'].values[WINDOW_SIZE:]
print(f'\nTrain: {times[0]} → {times[train_end-1]}')
print(f'Val:   {times[train_end]} → {times[val_end-1]}')
print(f'Test:  {times[val_end]} → {times[-1]}')

## 6. Modèle LSTM

In [ ]:
def build_lstm_model(window_size, n_features, name='lstm'):
    """LSTM pour classification binaire direction SOL."""
    model = keras.Sequential([
        layers.Input(shape=(window_size, n_features)),
        layers.LSTM(128, return_sequences=True),
        layers.Dropout(0.3),
        layers.LSTM(64),
        layers.Dropout(0.3),
        layers.Dense(32, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(1, activation='sigmoid')
    ], name=name)

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy', keras.metrics.AUC(name='auc')]
    )

    return model

model_15m = build_lstm_model(WINDOW_SIZE, n_features, name='lstm_15m')
model_15m.summary()

## 7. Entraînement

In [ ]:
BATCH_SIZE = 512
EPOCHS = 20

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_auc',
        patience=5,
        mode='max',
        restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6
    )
]

print(f'Training LSTM 15min — {X_train.shape[0]:,} samples, {EPOCHS} epochs, batch {BATCH_SIZE}')

history_15m = model_15m.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

## 8. Évaluation

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, roc_curve

# Test set evaluation
test_loss, test_acc, test_auc = model_15m.evaluate(X_test, y_test, verbose=0)
print(f'Test — Loss: {test_loss:.4f}, Accuracy: {test_acc:.4f}, AUC: {test_auc:.4f}')

# Predictions
y_pred_prob = model_15m.predict(X_test).flatten()
y_pred = (y_pred_prob > 0.5).astype(int)

print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Baisse', 'Hausse']))

print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred))

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history_15m.history['loss'], label='train')
axes[0].plot(history_15m.history['val_loss'], label='val')
axes[0].set_title('Loss')
axes[0].legend()

axes[1].plot(history_15m.history['accuracy'], label='train')
axes[1].plot(history_15m.history['val_accuracy'], label='val')
axes[1].set_title('Accuracy')
axes[1].legend()

axes[2].plot(history_15m.history['auc'], label='train')
axes[2].plot(history_15m.history['val_auc'], label='val')
axes[2].set_title('AUC')
axes[2].legend()

plt.tight_layout()
plt.show()

# ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)
plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, label=f'AUC = {test_auc:.3f}')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - LSTM 15min')
plt.legend()
plt.show()

## 9. Entraîner les 2 autres horizons

In [ ]:
# Horizon 5min
y_train_5m = labels_5m[WINDOW_SIZE:WINDOW_SIZE+train_end]
y_val_5m = labels_5m[WINDOW_SIZE+train_end:WINDOW_SIZE+val_end]
y_test_5m = labels_5m[WINDOW_SIZE+val_end:WINDOW_SIZE+len(X_windows)]

model_5m = build_lstm_model(WINDOW_SIZE, n_features, name='lstm_5m')
print(f'\nTraining LSTM 5min...')
history_5m = model_5m.fit(
    X_train, y_train_5m,
    validation_data=(X_val, y_val_5m),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

test_loss_5m, test_acc_5m, test_auc_5m = model_5m.evaluate(X_test, y_test_5m, verbose=0)
print(f'Test 5m — Accuracy: {test_acc_5m:.4f}, AUC: {test_auc_5m:.4f}')

In [ ]:
# Horizon 1h
y_train_60m = labels_60m[WINDOW_SIZE:WINDOW_SIZE+train_end]
y_val_60m = labels_60m[WINDOW_SIZE+train_end:WINDOW_SIZE+val_end]
y_test_60m = labels_60m[WINDOW_SIZE+val_end:WINDOW_SIZE+len(X_windows)]

model_60m = build_lstm_model(WINDOW_SIZE, n_features, name='lstm_60m')
print(f'\nTraining LSTM 1h...')
history_60m = model_60m.fit(
    X_train, y_train_60m,
    validation_data=(X_val, y_val_60m),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

test_loss_60m, test_acc_60m, test_auc_60m = model_60m.evaluate(X_test, y_test_60m, verbose=0)
print(f'Test 1h — Accuracy: {test_acc_60m:.4f}, AUC: {test_auc_60m:.4f}')

## 10. Résumé des 3 modèles

In [ ]:
print('═' * 50)
print('RÉSUMÉ DES MODÈLES')
print('═' * 50)
print(f'LSTM 5min  — Accuracy: {test_acc_5m:.4f}, AUC: {test_auc_5m:.4f}')
print(f'LSTM 15min — Accuracy: {test_acc:.4f}, AUC: {test_auc:.4f}')
print(f'LSTM 1h    — Accuracy: {test_acc_60m:.4f}, AUC: {test_auc_60m:.4f}')
print('═' * 50)

## 11. Export en TensorFlow.js

In [ ]:
!pip install tensorflowjs -q

In [ ]:
import tensorflowjs as tfjs

# Sauvegarder les 3 modèles en format TF.js
os.makedirs('/tmp/models', exist_ok=True)

for name, model in [('lstm_5m', model_5m), ('lstm_15m', model_15m), ('lstm_60m', model_60m)]:
    out_path = f'/tmp/models/{name}'
    tfjs.converters.save_keras_model(model, out_path)
    print(f'Exported {name} → {out_path}')

!ls -la /tmp/models/*/

In [ ]:
# Upload modèles sur GCS
!gsutil -m cp -r /tmp/models/* {BUCKET}/models/

print('\nModèles uploadés sur GCS:')
!gsutil ls {BUCKET}/models/

## 12. Sauvegarder aussi en format Keras (.h5)

Pour pouvoir reprendre l'entraînement plus tard ou fine-tuner.

In [ ]:
for name, model in [('lstm_5m', model_5m), ('lstm_15m', model_15m), ('lstm_60m', model_60m)]:
    model.save(f'/tmp/models/{name}.keras')
    print(f'Saved {name}.keras')

!gsutil -m cp /tmp/models/*.keras {BUCKET}/models/
print('\nKeras models uploaded to GCS')